# CSC 591/791 Homework 2: GCN vs. GraphSAGE & Scalability Analysis

This notebook runs all experiments for Homework 2 on Google Colab with GPU acceleration:
- **Task 1**: Full-Batch GCN Reproduction, Convergence & Ablations (`ogbn-arxiv`)
- **Task 2**: Mini-Batch GraphSAGE & Neighborhood Sampling (`ogbn-products`)
- **Task 3**: Comparative Profiling (Peak Memory & Runtime) and Plot Generation

## 1. Setup & Environment
Make sure GPU runtime is selected: **Runtime > Change runtime type > GPU (T4 or A100)**.

In [ ]:
!nvidia-smi

In [ ]:
# Install PyTorch Geometric and OGB
!pip install torch-geometric ogb psutil seaborn

## 2. Clone Repository or Set Working Directory
If you haven't cloned your GitHub repo yet, run the cell below:

In [ ]:
import os
if not os.path.exists('CSC591-038-_hw2'):
    !git clone https://github.com/vaishvi-code/CSC591-038-_hw2.git
    %cd CSC591-038-_hw2
else:
    %cd CSC591-038-_hw2
    !git pull origin main

!mkdir -p results/metrics results/plots

## 3. Task 1: Full-Batch GCN on ogbn-arxiv
### 3.1 GCN Baseline (100 Epochs)

In [ ]:
!python src/train_gcn.py --num_layers 3 --lr 0.01 --epochs 100 --exp_name gcn_baseline

### 3.2 GCN Model Depth Analysis (L = 2, 3, 4, 5, 6)

In [ ]:
for depth in [2, 4, 5, 6]:
    print(f"\n>>> Running GCN with Depth {depth}...")
    !python src/train_gcn.py --num_layers {depth} --lr 0.01 --epochs 80 --exp_name gcn_depth_{depth}

### 3.3 GCN Learning Rate Analysis (lr = 0.001, 0.005, 0.05)

In [ ]:
for lr in [0.001, 0.005, 0.05]:
    print(f"\n>>> Running GCN with LR {lr}...")
    !python src/train_gcn.py --num_layers 3 --lr {lr} --epochs 80 --exp_name gcn_lr_{lr}

## 4. Task 2: Mini-Batch GraphSAGE on ogbn-products
### 4.1 GraphSAGE Baseline (Fanout: 15, 10, 5; Batch Size: 1024)

In [ ]:
!python src/train_sage.py --num_layers 3 --sizes 15,10,5 --batch_size 1024 --epochs 15 --exp_name sage_baseline

### 4.2 Batch Size Analysis (B = 512, 2048, 4096)

In [ ]:
for bs in [512, 2048, 4096]:
    print(f"\n>>> Running GraphSAGE with Batch Size {bs}...")
    !python src/train_sage.py --num_layers 3 --sizes 15,10,5 --batch_size {bs} --epochs 12 --exp_name sage_batch_{bs}

### 4.3 Neighbor Sample Size Analysis (Fanouts: 5,5,5 vs 10,10,5)

In [ ]:
for sizes in ['5,5,5', '10,10,5']:
    clean_name = sizes.replace(',', '_')
    print(f"\n>>> Running GraphSAGE with Fanout {sizes}...")
    !python src/train_sage.py --num_layers 3 --sizes {sizes} --batch_size 1024 --epochs 12 --exp_name sage_sampling_{clean_name}

## 5. Task 3: Generate Figures & Comparative Profiling

In [ ]:
!python src/plot_results.py

### Display Generated Figures

In [ ]:
from IPython.display import Image, display
import glob

for img_file in sorted(glob.glob('results/plots/*.png')):
    print(f"Displaying {img_file}:")
    display(Image(filename=img_file))


## 6. Download Results Zip

In [ ]:
from google.colab import files
!zip -r results.zip results/
files.download('results.zip')
